# Work Script: test_limit_debug
Converted from `sft/work/test_limit_debug.py`.


In [ ]:
"""Debug test_limit_dialogues failure."""
import sys, os, tempfile, csv
from pathlib import Path
import pandas as pd
from typing import Optional

LIMIT_DIALOGUES = None

_DTYPES = {
    "dialogue_id": "str", "turn_id": "Int64", "speaker": "str",
    "utterance": "object", "source_dataset": "str", "original_id": "str",
    "extracted_entities": "object", "entity_labels": "object", "entity_count": "Int64",
}

In [ ]:
def load_turns(csv_path, usecols=None, chunksize=50_000, limit=None):
    effective_limit = limit if limit is not None else LIMIT_DIALOGUES
    print(f"  effective_limit={effective_limit}, limit={limit}, LIMIT_DIALOGUES={LIMIT_DIALOGUES}")
    cols = usecols or list(_DTYPES.keys())
    dtype = {c: _DTYPES[c] for c in cols if c in _DTYPES}
    chunks = []
    seen_dialogues: set = set()
    reader = pd.read_csv(csv_path, usecols=cols, dtype=dtype, chunksize=chunksize,
                          encoding="utf-8", low_memory=False)
    for chunk in reader:
        if effective_limit is not None:
            # Add new dialogue IDs only up to the limit
            for did in chunk["dialogue_id"].dropna().unique():
                if len(seen_dialogues) < effective_limit:
                    seen_dialogues.add(did)
            # Keep only rows whose dialogue is within the accepted set
            chunk = chunk[chunk["dialogue_id"].isin(seen_dialogues)]
            if len(chunk):
                chunks.append(chunk)
            if len(seen_dialogues) >= effective_limit:
                break
        else:
            chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)

In [ ]:
with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "test.csv"
    rows = []
    for d in range(10):
        did = f"dia_{d:04d}"
        for t in range(2):
            rows.append({"dialogue_id": did, "turn_id": t, "speaker": ["user","assistant"][t%2],
                         "utterance": f"text {d} {t}", "source_dataset": "MedDialog",
                         "original_id": str(d), "extracted_entities": "", "entity_labels": "", "entity_count": 0})
    with open(p, "w", encoding="utf-8", newline="\n") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        for r in rows:
            w.writerow(r)

    df = load_turns(p, limit=3)
    n = df["dialogue_id"].nunique()
    print(f"limit=3 → {n} unique dialogues (expected <=3)")
    assert n <= 3, f"FAIL: {n} > 3"

    df2 = load_turns(p, limit=None)
    n2 = df2["dialogue_id"].nunique()
    print(f"limit=None → {n2} unique dialogues (expected 10)")
    assert n2 == 10, f"FAIL: {n2} != 10"
    print("PASS")

In [ ]:
print("RESULT: PASS test_limit_debug.py notebook compiled successfully")